## Imports

In [2]:
# basic imports for data processing and plotting
import numpy as np
import pandas as pd
import logging as log
import os
import matplotlib.pyplot as plt
import unidecode


# imports for data loading
from nba_api.stats.endpoints import TeamGameLogs, PlayerGameLogs
import kagglehub

# sklearn imports for data splitting and model training
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_curve, balanced_accuracy_score
from sklearn.ensemble import GradientBoostingClassifier

from injury_preprocess import preprocess_injury_data



/opt/miniconda3/envs/dopp_as2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'utils'

## Define Constants

In [3]:
DATA_DIR = "data"
TEAMLOG_DATA = os.path.join(DATA_DIR, "team_data")
PLAYERLOG_DATA = os.path.join(DATA_DIR, "player_data")
INJURY_DATA = os.path.join(DATA_DIR, "injury_data")
PLOTS_DIR = "plots"
SEASONS = ["2016-17", "2017-18", "2018-19", "2019-20", "2020-21", "2021-22", "2022-23", "2023-24"]
DEFAULT_COLUMNS = ['SEASON_YEAR', 'TEAM_ID', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'WL', 'MATCHUP']
# NUMERIC_COLUMNS = ['PTS', 'PLUS_MINUS', 'FG_PCT', 'FGM', 'OREB', 'DREB', 'AST', 'BLK']
NUMERIC_COLUMNS = ['FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB',
                   'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS']

WIN_PCT_COLUMN = 'WIN_PCT'
AGG_WINDOW_SIZES = [5, 15]

## Define helper functions

In [5]:
def check_create_dir(path):
    if os.path.exists(path):
        print(f"Directory {path} already exists.")
    else:
        os.makedirs(path)
        print(f"Directory {path} created.")

def load_from_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df


## Data Loading

We load the data from the nba_api and kaggle.

In [13]:
log.basicConfig(level=log.INFO,
                format='%(asctime)s: %(levelname)s: %(message)s',
                datefmt='%Y-%m-%d %H:%M:%S')


def load_injury_data() -> pd.DataFrame:
    # Download latest version
    path = kagglehub.dataset_download("jacquesoberweis/2016-2025-nba-injury-data")

    print("Path to dataset files:", path)
    df = pd.read_csv(os.path.join(path, "injury_data.csv"))
    return df


def load_playerlogs(seasons: list[str]) -> pd.DataFrame:
    df = pd.DataFrame()
    for season in seasons:
        gamedatapull = PlayerGameLogs(
            league_id_nullable="00",  # nba 00, g_league 20, wnba 10
            season_type_nullable="Regular Season",  # Regular Season, Playoffs, Pre Season
            season_nullable=season,
        )

        df_season = gamedatapull.get_data_frames()[0]
        df = pd.concat([df, df_season])
    return df


def load_teamlogs(seasons: list[str]) -> pd.DataFrame:
    df = pd.DataFrame()
    for season in seasons:
        gamedatapull = TeamGameLogs(
            league_id_nullable="00",  # nba 00, g_league 20, wnba 10
            season_type_nullable="Regular Season",  # Regular Season, Playoffs, Pre Season
            season_nullable=season,
        )

        df_season = gamedatapull.get_data_frames()[0]
        df = pd.concat([df, df_season])
    return df



team_data = load_teamlogs(SEASONS)
#team_data.to_csv(os.path.join(TEAMLOG_DATA, "team_data.csv"), index=False)
log.info("Team data loaded into team_data")
log.info(team_data.head())

player_data = load_playerlogs(SEASONS)
#player_data.to_csv(os.path.join(PLAYERLOG_DATA, "player_data.csv"), index=False)
log.info("Player data loaded")
log.info(player_data.head())

injury_data = load_injury_data()
#injury_data.to_csv(os.path.join(INJURY_DATA, "injury_data.csv"), index=False)
log.info("Injury data loaded")
log.info(injury_data.head())


2025-01-26 13:26:59: INFO: Team data loaded into team_data
2025-01-26 13:26:59: INFO:   SEASON_YEAR     TEAM_ID TEAM_ABBREVIATION          TEAM_NAME     GAME_ID  \
0     2016-17  1610612743               DEN     Denver Nuggets  0021601225   
1     2016-17  1610612752               NYK    New York Knicks  0021601220   
2     2016-17  1610612763               MEM  Memphis Grizzlies  0021601223   
3     2016-17  1610612761               TOR    Toronto Raptors  0021601218   
4     2016-17  1610612737               ATL      Atlanta Hawks  0021601226   

             GAME_DATE      MATCHUP WL   MIN  FGM  ...  AST_RANK  TOV_RANK  \
0  2017-04-12T00:00:00    DEN @ OKC  W  48.0   39  ...      1222      1857   
1  2017-04-12T00:00:00  NYK vs. PHI  W  48.0   41  ...      2076      2148   
2  2017-04-12T00:00:00  MEM vs. DAL  L  48.0   33  ...      1933      1413   
3  2017-04-12T00:00:00    TOR @ CLE  W  48.0   40  ...      1584       178   
4  2017-04-12T00:00:00    ATL @ IND  L  48.0   30  ... 

Path to dataset files: /Users/filipfaber/.cache/kagglehub/datasets/jacquesoberweis/2016-2025-nba-injury-data/versions/5


## Preprocessing and Feature Engineering

### Proprocess Team Data

In [ ]:
def convert_dtypes_teamlogs(df: pd.DataFrame, numeric_cols=None) -> pd.DataFrame:
    df = df.copy()
    df['SEASON_YEAR'] = df['SEASON_YEAR'].str[:4].astype(int)
    df['TEAM_NAME'] = df['TEAM_NAME'].astype('string')
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df['HOME'] = df['MATCHUP'].str.contains(r'\bvs\b').astype(int)
    df.drop(columns=['MATCHUP'], inplace=True)
    df['WL'] = df['WL'].map({'W': 1, 'L': 0}).astype(int)
    if numeric_cols:
        for col in numeric_cols:
            if col in df.columns:
                if (df[col].dtype == 'int64' or df[col].dtype == 'float64') and col in df.columns:
                    df[col] = df[col].astype(float)
        return df
    else:
        return df



def compute_rolling_values(df: pd.DataFrame,
                           window_size: int,
                           agg_function=lambda x: np.mean(x),
                           group: list[str] = ['TEAM_ID', 'SEASON_YEAR']) -> pd.DataFrame:
    df = df.copy()
    for col in NUMERIC_COLUMNS:
        if col in df.columns:
            colname = col + '_AGG_' + str(window_size)
            df[colname] = df.groupby(group)[col].transform(
                lambda x: x.shift(1).rolling(window=window_size, min_periods=1).agg(agg_function))
    return df


def aggregate_team_stats(df: pd.DataFrame) -> pd.DataFrame:
    # sort by date
    df = df.sort_values(by=['GAME_DATE'])

    for window_size in AGG_WINDOW_SIZES:
        df = compute_rolling_values(df, window_size, agg_function=lambda x: np.mean(x))

    df = df.drop(columns=NUMERIC_COLUMNS, inplace=False)
    # Calculate season long win percentage
    df[WIN_PCT_COLUMN] = df.groupby(['TEAM_ID', 'SEASON_YEAR'])['WL'].transform(
        lambda x: x.shift(1).rolling(window=9999, min_periods=1).mean())

    # Drop first window_size rows for each team
    df = df.groupby(['TEAM_ID', 'SEASON_YEAR']).apply(
        lambda x: x.iloc[window_size:],
        include_groups=False
    ).reset_index(['TEAM_ID', 'SEASON_YEAR'], drop=False)

    return df

def combine_on_gameid(df) -> pd.DataFrame:
    if 'TEAM_NAME' in df.columns:
        df = df.drop(columns=['TEAM_NAME'], inplace=False)

    df_HOME = df[df['HOME'] == 1]
    df_AWAY = df[df['HOME'] == 0]
    df_HOME = df_HOME.drop(columns=['HOME'], inplace=False)
    df_AWAY = df_AWAY.drop(columns=['HOME', 'SEASON_YEAR', 'GAME_DATE', 'WL'], inplace=False)

    df_merged = df_HOME.merge(df_AWAY, on='GAME_ID', suffixes=('_HOME', '_AWAY'))

    #df_merged.set_index(['SEASON_YEAR', 'GAME_ID', 'GAME_DATE', 'TEAM_ID_HOME', 'TEAM_ID_AWAY'], inplace=True)

    return df_merged


def preprocess_teamlogs(df=None) -> pd.DataFrame:
    if df is None:
        log.info("Loading data from csv")
        df = load_from_csv(os.path.join(TEAMLOG_DATA, "team_data.csv"))
    log.info(df.head())

    log.info(df['TEAM_NAME'].unique())

    # drop variables that end with "_RANK"
    df = df.loc[:, ~df.columns.str.endswith('_RANK')]
    df = df.drop(columns=['AVAILABLE_FLAG'], inplace=False)

    # create data with all columns converted to appropriate data types
    df_team_converted = convert_dtypes_teamlogs(df)
    #df_full_converted.to_csv(os.path.join(TEAMLOG_DATA, 'team_data_full_converted.csv'), index=False)

    # create mapping from team name to team id and save to json file
    team_mapping = df_team_converted[['TEAM_NAME', 'TEAM_ID']].drop_duplicates()
    team_mapping.to_json(os.path.join('mappings', 'team_id_mapping.json'), orient='records')

    COLUMNS = DEFAULT_COLUMNS + NUMERIC_COLUMNS
    df = df[COLUMNS]

    log.info("Checking for missing values")
    missing_values = df.isnull().sum()
    log.info(missing_values)

    df = convert_dtypes_teamlogs(df, NUMERIC_COLUMNS)
    df.sort_values(by=['GAME_DATE', 'GAME_ID'], inplace=True)
    df.to_csv(os.path.join(TEAMLOG_DATA, 'team_data_converted.csv'), index=False)

    df = aggregate_team_stats(df)
    df.to_csv(os.path.join(TEAMLOG_DATA, 'team_data_aggregated.csv'), index=False)

    df = combine_on_gameid(df)
    return df, df_team_converted 
    df.to_csv(os.path.join(TEAMLOG_DATA, 'team_data_combined.csv'), index=True)

df_team, df_team_converted = preprocess_teamlogs(team_data)
df_team.head()


2025-01-26 13:32:58: INFO:   SEASON_YEAR     TEAM_ID TEAM_ABBREVIATION          TEAM_NAME     GAME_ID  \
0     2016-17  1610612743               DEN     Denver Nuggets  0021601225   
1     2016-17  1610612752               NYK    New York Knicks  0021601220   
2     2016-17  1610612763               MEM  Memphis Grizzlies  0021601223   
3     2016-17  1610612761               TOR    Toronto Raptors  0021601218   
4     2016-17  1610612737               ATL      Atlanta Hawks  0021601226   

             GAME_DATE      MATCHUP WL   MIN  FGM  ...  AST_RANK  TOV_RANK  \
0  2017-04-12T00:00:00    DEN @ OKC  W  48.0   39  ...      1222      1857   
1  2017-04-12T00:00:00  NYK vs. PHI  W  48.0   41  ...      2076      2148   
2  2017-04-12T00:00:00  MEM vs. DAL  L  48.0   33  ...      1933      1413   
3  2017-04-12T00:00:00    TOR @ CLE  W  48.0   40  ...      1584       178   
4  2017-04-12T00:00:00    ATL @ IND  L  48.0   30  ...      1762      2250   

   STL_RANK  BLK_RANK  BLKA_RANK  P

,TEAM_ID_HOME,SEASON_YEAR,GAME_ID,GAME_DATE,WL,FGM_AGG_5_HOME,FGA_AGG_5_HOME,FG_PCT_AGG_5_HOME,FG3M_AGG_5_HOME,FG3A_AGG_5_HOME,...,AST_AGG_15_AWAY,TOV_AGG_15_AWAY,STL_AGG_15_AWAY,BLK_AGG_15_AWAY,BLKA_AGG_15_AWAY,PF_AGG_15_AWAY,PFD_AGG_15_AWAY,PTS_AGG_15_AWAY,PLUS_MINUS_AGG_15_AWAY,WIN_PCT_AWAY
0,1610612737,2016,0021600288,2016-12-02,0,36.4,86.0,0.4220,8.2,26.8,...,20.600000,10.933333,6.533333,4.133333,5.133333,17.600000,16.533333,99.200000,0.666667,0.500000
1,1610612737,2016,0021600307,2016-12-05,0,37.4,87.6,0.4270,8.2,28.2,...,23.533333,15.066667,6.933333,5.200000,5.066667,19.933333,19.133333,109.266667,2.133333,0.619048
2,1610612737,2016,0021600324,2016-12-07,1,36.4,85.4,0.4268,7.2,25.6,...,19.933333,12.000000,7.533333,5.600000,6.066667,20.133333,18.466667,97.800000,-2.400000,0.333333
3,1610612737,2016,0021600369,2016-12-13,0,36.8,85.0,0.4338,7.2,25.4,...,20.400000,13.666667,6.933333,5.866667,5.733333,18.200000,18.000000,94.666667,-2.733333,0.400000
4,1610612737,2016,0021600401,2016-12-17,0,41.6,84.8,0.4896,9.0,25.6,...,22.200000,12.666667,5.466667,5.066667,5.733333,16.133333,20.666667,102.000000,-1.000000,0.518519


### Preprocess Player Data

In [14]:
def convert_dtypes_playerlogs(df: pd.DataFrame, numeric_cols=None) -> pd.DataFrame:
    df = df.copy()
    df = df[['SEASON_YEAR', 'PLAYER_ID', 'PLAYER_NAME', 'GAME_DATE', 'MIN', 'PTS', 'PLUS_MINUS']]
    df['SEASON_YEAR'] = df['SEASON_YEAR'].str[:4].astype(int)
    df['PLAYER_ID'] = df['PLAYER_ID'].astype(int)
    df['PLAYER_NAME'] = df['PLAYER_NAME'].astype('string')
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df['MIN'] = df['MIN'].astype(float)
    df['PTS'] = df['PTS'].astype(float)
    df['PLUS_MINUS'] = df['PLUS_MINUS'].astype(float)
    return df


def preprocess_stats(df=None):
    if df is None:
        log.info("Loading data from csv")
        df = load_from_csv(os.path.join(PLAYERLOG_DATA, "player_data.csv"))
    df["Season"] = df["SEASON_YEAR"].apply(lambda x: int(x[:4]))
    df["Team"] = df["TEAM_ABBREVIATION"].astype(str)
    df["Player"] = df["PLAYER_NAME"].astype(str)
    df["Player"] = df["Player"].apply(lambda x: unidecode.unidecode(x))

    df.drop(columns=["SEASON_YEAR", "PLAYER_NAME"], inplace=True)
    df.columns = df.columns.str.upper()
    return df
    df.to_csv(os.path.join(PLAYERLOG_DATA, "player_data_cleaned.csv"), index=False)

""" def preprocess_player_logs():

    def aggregate_player_stats(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df = df.sort_values(by=['GAME_DATE', 'PLAYER_NAME'])
        df = compute_rolling_values(df, 9999, agg_function=lambda x: np.mean(x), group=['PLAYER_ID', 'SEASON_YEAR'])

        return df

    log.info("Loading data from csv")
    df = load_from_csv(os.path.join(PLAYERLOG_DATA, "player_data.csv"))
    log.info(df.head())
    create_player_mapping(df)

    # drop variables that end with "_RANK"
    df = convert_dtypes_playerlogs(df)

    # create data with all columns converted to appropriate data types
    df.to_csv(os.path.join(PLAYERLOG_DATA, 'player_data_converted.csv'), index=False)

    df = aggregate_player_stats(df)
    df.to_csv(os.path.join(PLAYERLOG_DATA, 'player_data_aggregated.csv'), index=False)


def create_player_mapping(df: pd.DataFrame):
    # create json file containing player name to player id mapping
    player_mapping = df[['PLAYER_NAME', 'PLAYER_ID']].drop_duplicates()
    player_mapping.to_json(os.path.join(DATA_DIR, 'player_mapping.json'), orient='records')
 """

df_player = preprocess_stats(player_data)
df_player.head()

,PLAYER_ID,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,...,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC,SEASON,TEAM,PLAYER
0,201582,Alexis,1610612740,NOP,New Orleans Pelicans,0021601230,2017-04-12T00:00:00,NOP @ POR,W,25.170000,...,10057,12157,2032,118,10793,1.0,25:10,2016,NOP,Alexis Ajinca
1,202683,Enes,1610612760,OKC,Oklahoma City Thunder,0021601225,2017-04-12T00:00:00,OKC vs. DEN,L,19.850000,...,5259,11408,2032,118,10793,1.0,19:51,2016,OKC,Enes Freedom
2,201580,JaVale,1610612744,GSW,Golden State Warriors,0021601229,2017-04-12T00:00:00,GSW vs. LAL,W,17.850000,...,4644,9459,2032,118,10056,1.0,17:51,2016,GSW,JaVale McGee
3,200768,Kyle,1610612761,TOR,Toronto Raptors,0021601218,2017-04-12T00:00:00,TOR @ CLE,W,17.150000,...,3615,13660,2032,118,10793,1.0,17:09,2016,TOR,Kyle Lowry
4,203490,Otto,1610612764,WAS,Washington Wizards,0021601221,2017-04-12T00:00:00,WAS @ MIA,L,14.683333,...,20002,13038,2032,118,13020,1.0,14:41,2016,WAS,Otto Porter Jr.


### Preprocess Injury Data

In [17]:
def preprocess_injury_data(df=None) -> pd.DataFrame:
    if df is None:
        log.info("Loading data from csv")
        df = load_from_csv(os.path.join(INJURY_DATA, "injury_data.csv"))

    df["Date"] = pd.to_datetime(
        df["Date"], format="%Y-%m-%d", errors="coerce"
    )

    players = []
    if "Relinquished" in df.columns:
        players = df["Relinquished"].dropna().unique().tolist()

    injury_start = df[df["Relinquished"].notna()][
        ["Date", "Team", "Relinquished", "Notes"]
    ].rename(columns={"Date": "Injury_Start", "Relinquished": "Player"})

    injury_end = df[df["Acquired"].notna()][
        ["Date", "Team", "Acquired"]
    ].rename(columns={"Date": "Injury_End", "Acquired": "Player"})

    result = []
    for player, group in injury_start.groupby("Player"):
        injury_group = group.sort_values(by="Injury_Start")
        recovery_group = injury_end[injury_end["Player"] == player].sort_values(
            by="Injury_End"
        )
        recovery_dates = iter(recovery_group["Injury_End"])

        current_recovery = None
        try:
            current_recovery = next(recovery_dates)
        except StopIteration:
            pass

        for i, (index, injury_row) in enumerate(injury_group.iterrows()):
            injury_note = injury_row["Notes"] if pd.notna(injury_row["Notes"]) else ""
            current_injury = injury_row["Injury_Start"]

            # Calculate season-ending date if applicable
            injury_end_date = pd.NaT
            if "(out for season)" in injury_note.lower():
                current_year = current_injury.year
                august_15_current_year = pd.Timestamp(f"{current_year}-08-15")

                if current_injury > august_15_current_year:
                    injury_end_date = pd.Timestamp(f"{current_year + 1}-08-15")
                else:
                    injury_end_date = august_15_current_year

            if i + 1 < len(injury_group):
                next_injury = injury_group.iloc[i + 1]["Injury_Start"]
                if current_recovery and current_recovery > next_injury:
                    result.append(
                        {
                            "Player": player,
                            "Team": injury_row["Team"],
                            "Injury_Start": current_injury,
                            "Injury_End": injury_end_date,
                            "Injury_Notes": injury_note,
                        }
                    )
                else:
                    end_date = (
                        injury_end_date
                        if not pd.isna(injury_end_date)
                        else current_recovery
                    )
                    result.append(
                        {
                            "Player": player,
                            "Team": injury_row["Team"],
                            "Injury_Start": current_injury,
                            "Injury_End": end_date,
                            "Injury_Notes": injury_note,
                        }
                    )
                    try:
                        current_recovery = next(recovery_dates)
                    except StopIteration:
                        current_recovery = None
            else:
                if current_recovery and current_recovery > current_injury:
                    end_date = current_recovery
                else:
                    end_date = (
                        injury_end_date if not current_recovery else current_recovery
                    )

                result.append(
                    {
                        "Player": player,
                        "Team": injury_row["Team"],
                        "Injury_Start": current_injury,
                        "Injury_End": end_date,
                        "Injury_Notes": injury_note,
                    }
                )

    # Convert to DataFrame and clean up
    result_df = pd.DataFrame(result)

    # Iterating backwards to combine injuries that are consecutive, and fixing those that are season ending without having knowledge of it at the time
    for i in range(len(result_df) - 1, -1, -1):
        # Check if current row has NaT in Injury_End
        if pd.isna(result_df.iloc[i]["Injury_End"]):
            current_injury_start = result_df.iloc[i]["Injury_Start"]
            current_player = result_df.iloc[i]["Player"]

            # Get the next August 15 after Injury_Start
            current_year = current_injury_start.year
            august_15_current_year = pd.Timestamp(f"{current_year}-08-15")

            if current_injury_start > august_15_current_year:
                default_injury_end = pd.Timestamp(f"{current_year + 1}-08-15")
            else:
                default_injury_end = august_15_current_year

            # Check if there's a row below (i < len(df) - 1) and if names match
            if (
                i < len(result_df) - 1
                and current_player == result_df.iloc[i + 1]["Player"]
            ):
                next_injury_start = result_df.iloc[i + 1]["Injury_Start"]
                next_august_15 = pd.Timestamp(f"{current_injury_start.year}-08-15")
                if current_injury_start > next_august_15:
                    next_august_15 = pd.Timestamp(
                        f"{current_injury_start.year + 1}-08-15"
                    )

                # Check if next injury starts before next August 15
                if next_injury_start <= next_august_15:
                    # Combine rows by taking end date from next row
                    result_df.at[i, "Injury_End"] = result_df.iloc[i + 1]["Injury_End"]
                    # Combine injury notes if they exist
                    if pd.notna(result_df.iloc[i + 1]["Injury_Notes"]):
                        result_df.at[i, "Injury_Notes"] = (
                            str(result_df.iloc[i]["Injury_Notes"])
                            + " ; "
                            + str(result_df.iloc[i + 1]["Injury_Notes"])
                        )
                    # Drop the next row as it's now combined
                    result_df = result_df.drop(index=result_df.index[i + 1])
                else:
                    # Set end date to next August 15
                    result_df.at[i, "Injury_End"] = default_injury_end
            else:
                # Set end date to next August 15
                result_df.at[i, "Injury_End"] = default_injury_end

    result_df["Team"] = result_df["Team"].astype(str)
    result_df["Player"] = result_df["Player"].astype(str).str.strip()
    result_df["Injury_Notes"] = result_df["Injury_Notes"].astype(str).str.strip()

    # Load mappings
    team_name_mapping = pd.read_json(os.path.join('mappings', 'team_name_mapping.json'))
    team_id_mapping = pd.read_json(os.path.join('mappings', 'team_id_mapping.json'))
    player_mapping = pd.read_json(os.path.join('mappings', 'player_mapping.json'))

    # Mapping Team Name to Team ID
    result_df = result_df.merge(team_name_mapping, left_on='Team', right_on='Team', how='left')
    result_df = result_df.drop(columns=['Team'], inplace=False)
    result_df.rename(columns={'TEAM_NAME': 'TEAM'}, inplace=True)
    result_df = result_df.merge(team_id_mapping, left_on='TEAM', right_on='TEAM_NAME', how='left')
    result_df = result_df.drop(columns=['TEAM', 'TEAM_NAME'], inplace=False)

    # from "Player" column remove string that are contained in brackets ()
    result_df["Player"] = result_df["Player"].str.replace(r"\(.*\)", "", regex=True)
    # Split a row where the name contatins "/" into two rows
    result_df = result_df.assign(
        Player=result_df["Player"].str.split("/")).explode("Player")
    result_df["Player"] = result_df["Player"].str.strip()

    # turn into datetime
    result_df["Injury_Start"] = pd.to_datetime(result_df["Injury_Start"])
    result_df["Injury_End"] = pd.to_datetime(result_df["Injury_End"])

    # rename columns into Caps
    result_df.columns = result_df.columns.str.upper()

    result_df = result_df.sort_values(["PLAYER", "INJURY_START"])
    result_df = result_df.reset_index(drop=True)


    #result_df.to_csv(os.path.join(INJURY_DATA, "injury_data_cleaned.csv"), index=False)
    return result_df

df_injury = preprocess_injury_data(injury_data)
df_injury.head()

,PLAYER,INJURY_START,INJURY_END,INJURY_NOTES,TEAM_ID
0,A.J. Green,2022-10-29,2022-11-16,placed on IL with fractured nose,1610612749
1,A.J. Green,2024-01-02,2024-01-04,placed on IL with nasal fracture,1610612749
2,A.J. Green,2024-03-24,2024-03-25,placed on IL with illness,1610612749
3,A.J. Green,2024-04-11,2024-04-20,placed on IL with sprained left ankle,1610612749
4,A.J. Green,2025-01-02,2025-01-04,placed on IL with lumbar injury,1610612749


### Merging Injury Data with Game data

In [19]:
def merge_games_injuries(df_team=None, df_player=None, df_injury=None) -> pd.DataFrame:
    log.info("Merging team data with injury data")
    if df_team is None:
        df_team = load_from_csv(os.path.join(TEAMLOG_DATA, 'team_data_combined.csv'))
    if df_player is None:
        df_player = load_from_csv(os.path.join(PLAYERLOG_DATA, 'player_advanced_cleaned.csv'))
    if df_injury is None:
        df_injury = load_from_csv(os.path.join(INJURY_DATA, 'injury_data_cleaned.csv'))
        
    # injury start in datetime format
    df_team['GAME_DATE'] = pd.to_datetime(df_team['GAME_DATE'])
    df_injury['INJURY_START'] = pd.to_datetime(df_injury['INJURY_START'])
    df_injury['INJURY_END'] = pd.to_datetime(df_injury['INJURY_END'])

    def list_injured_players(row, team_col):
        injured_players = df_injury[
            (df_injury['TEAM_ID'] == row[team_col]) &
            (df_injury['INJURY_START'] <= row['GAME_DATE']) &
            (df_injury['INJURY_END'] >= row['GAME_DATE'])
        ]['PLAYER'].tolist()
        return injured_players

    df_team['INJURED_PLAYERS_HOME'] = df_team.apply(list_injured_players, axis=1, team_col='TEAM_ID_HOME')
    df_team['INJURED_PLAYERS_AWAY'] = df_team.apply(list_injured_players, axis=1, team_col='TEAM_ID_AWAY')

    """ def get_player_stats(row, player_col):
        stats = []
        for player_name in row[player_col]:
            player_stats = df_player[
                (df_player['PLAYER'] == player_name) &
                (df_player['SEASON'] < row['SEASON_YEAR'].year)
            ].sort_values(by='SEASON', ascending=False).head(1)
            if not player_stats.empty:
                stats.append(player_stats.iloc[0]['WS/48'])

        return np.sum(stats) """

    df_player = load_from_csv(os.path.join(PLAYERLOG_DATA, 'player_data_cleaned.csv'))
    df_player['GAME_DATE'] = pd.to_datetime(df_player['GAME_DATE'])

    def get_player_stats(row, player_col):
        stats = []
        for player_name in row[player_col]:
            player_stats = df_player[
                (df_player['PLAYER'] == player_name) &
                (df_player['GAME_DATE'] < row['GAME_DATE'])
            ].sort_values(by='GAME_DATE', ascending=False).head(20)
            if not player_stats.empty:
                mins = player_stats['MIN'].mean()
                pts = player_stats['PTS'].mean()
                reb = player_stats['REB'].mean()
                ast = player_stats['AST'].mean()
                stl = player_stats['STL'].mean()
                blk = player_stats['BLK'].mean()
                player_importance = mins+pts+reb+ast+stl+blk
                stats.append(player_importance)

        return np.sum(stats)

    # df_team = df_team.explode('INJURED_PLAYERS_HOME')
    df_team['INJURED_PLAYERS_HOME_STATS'] = df_team.apply(get_player_stats, axis=1, player_col='INJURED_PLAYERS_HOME')
    # df_team = df_team.explode('INJURED_PLAYERS_AWAY')
    df_team['INJURED_PLAYERS_AWAY_STATS'] = df_team.apply(get_player_stats, axis=1, player_col='INJURED_PLAYERS_AWAY')

    df_team = df_team.drop(columns=['INJURED_PLAYERS_HOME', 'INJURED_PLAYERS_AWAY'], inplace=False)
    
    return df_team
    df_team.to_csv(os.path.join(TEAMLOG_DATA, 'team_data_combined_injuries.csv'), index=True)

df_full = merge_games_injuries(df_team, df_player, df_injury)

2025-01-26 13:33:12: INFO: Merging team data with injury data


## Data Exploration